In [ ]:
import ee, eemont
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
from pygee import gee

# 1. Load GEE datasets

In [ ]:
fc_lia = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_with_buffer200')
geom_lia = ee.Geometry.convexHull(fc_lia.geometry())

# 2. Get all S2 indices 

## 2.1 Parameters

Indices parameters

In [ ]:
day_beg = ee.Date('2017-07-15')
day_end = ee.Date('2023-10-15')

In [ ]:
month_list = [7,8,9]

In [ ]:
cloud_thresh = 60

In [ ]:
indices_classif = ["NARI", "NCRI"]

Scale and projection

In [ ]:
scale = 20
projection = "EPSG:3035"

## 2.1 Monthly median over summer

In [ ]:
s2_indices_ts = (ee.ImageCollection('COPERNICUS/S2_SR')
                    .filter(ee.Filter.calendarRange(day_beg.getRelative("day", "year"),
                                                    day_end.getRelative("day", "year"),
                                                    'day_of_year'))
                    .filter(ee.Filter.calendarRange(2017, 2023, 'year'))
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_thresh)) 
                    .filterBounds(geom_lia)
                    .scaleAndOffset()
                    .maskClouds(scaledImage=True)
                    .map(gee.add_s2_nari)
                    .map(gee.add_s2_ncri)
                 ).select(indices_classif)

In [ ]:
s2_ts_mm = gee.ic_monthly_median(s2_indices_ts, 
                                 month_list=month_list, 
                                 first_day_of_month=15, 
                                 band_names=indices_classif)

In [ ]:
geemap.ee_export_image_to_asset(s2_ts_mm.clip(fc_lia), 
                                assetId="users/aguerou/ice_and_life/carto_h1b/indices/s2_indices_nari_ncri_median_ts_lia_Reinthaler_Mourey_with_buffer200",
                                region=geom_lia,
                                scale=scale,
                                crs=projection,
                                maxPixels=1e9)